# Lab 6 — Segment the Customers
**SDA-AIE-111 · Day 4 · Hour 3 · 50 minutes · pairs**

**Objective.** Deliver a named, profiled, null-tested customer segmentation with a PCA segment map; validate segments *externally* by churn-rate spread; add `segment` as a churn-model feature and measure the effect honestly.

**The module's conscience:** k-means always returns k clusters — even on pure noise. Every claim below must survive the null test.


In [ ]:
# --- Course setup (identical in every lab) -----------------------------------
import sys, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
RANDOM_STATE = 42

def find_repo_root() -> Path:
    """Locate the course repo root by looking for data/manafeth_customers.parquet."""
    for cand in [Path.cwd(), *Path.cwd().parents]:
        if (cand / "data" / "manafeth_customers.parquet").exists():
            return cand
    raise FileNotFoundError(
        "Course data not found — copy the Manafeth data package files into <repo>/data/")

ROOT = find_repo_root()
DATA = ROOT / "data"
sys.path.insert(0, str(ROOT / "src"))
print("repo root:", ROOT)

In [ ]:
from sklearn.model_selection import train_test_split
from manafeth.features import (build_preprocessor, build_feature_table,
                               MODEL_NUM_COLS, CAT_COLS, LEAKY)
from manafeth.evaluation import CV, evaluate, recall_at_budget, precision_at_budget

customers = pd.read_parquet(DATA / "manafeth_customers.parquet")
orders = pd.read_parquet(DATA / "manafeth_orders.parquet")
full = build_feature_table(customers, orders, pd.Timestamp("2025-11-01"))

X_all = full.drop(columns=["churned_30d", "customer_id", *LEAKY])
y_all = full["churned_30d"]
X_train, X_test, y_train, y_test = train_test_split(
    X_all, y_all, test_size=0.20, stratify=y_all, random_state=RANDOM_STATE)
print("train:", X_train.shape, "| churn:", round(y_train.mean(), 3))

## Task 1 — Scale, restart, choose k three ways *(10 min)*

Money features have a savage right tail (bulk-order households) — log them **before** scaling, or one axis owns every distance.


In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler

SEG_COLS = ["orders_per_month", "avg_basket_sar", "days_since_last_order",
            "tenure_months", "distinct_categories", "promo_usage_rate",
            "ramadan_order_share", "weekend_order_share"]

# TODO: log1p the money column, then StandardScaler -> Z
# TODO: sweep k=2..10; record inertia (elbow) and silhouette (sample_size=10_000, seeded)
# TODO: plot both panels; record YOUR k vote BEFORE profiling


## Task 2 — Profile and name *(10 min)*

A segment nobody can name is a segment nobody will use. Name every segment in ≤ 3 words; an unnameable segment is a finding (revisit k), not a failure.


In [ ]:
# TODO: fit the final k-means (your k, seeded); build the profile table:
#       size, orders_pm, basket, recency, ramadan_share, weekend_share, churn_rate
# TODO: sort by churn_rate and NAME every segment in <= 3 words


## Task 3 — PCA: budget sheet, map, loadings *(10 min)*

Compress, view, interpret — in that order. The 2-D map is a *view*, not proof.


In [ ]:
from sklearn.decomposition import PCA

# TODO: fit PCA on Z; print cumulative explained variance (what do PC1+PC2 carry?)
# TODO: scatter the first two PCs coloured by segment
# TODO: print the PC1/PC2 loadings and write a one-line reading of each axis


## Task 4 — The null test *(5 min)*

Would we have "found" structure in pure noise? Report both numbers; the gap IS the claim.


In [ ]:
# TODO: k-means (k=5) on standard-normal noise of the same shape
# TODO: compare silhouettes (real vs noise) — report BOTH numbers


## Task 5 — External usefulness: segment as a feature *(10 min)*

The churn-rate spread already validated the segments externally. Now the harder test: does `segment` help the supervised champion?


In [ ]:
# TODO: add the segment label (as a string/categorical) to X_train
# TODO: re-run the Lab 5 XGBoost through evaluate() with "segment" as an extra
#       categorical column — record the delta and interpret it honestly


**Commit:** `feat(lab6): named segments, null-tested, churn-validated`
